In [ ]:
# Replace entire Step 1 with this
!pip install -q huggingface_hub

# Install llama-cpp-python with pre-built CUDA wheels (no compile needed)
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
  --no-cache-dir -q

# Still need convert_hf_to_gguf.py — clone just the scripts, skip building
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

print('✅ Done — no compilation required')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 153.2 MB/s eta 0:00:00
Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3171, done.
remote: Counting objects: 100% (3171/3171), done.
remote: Compressing objects: 100% (2505/2505), done.
remote: Total 3171 (delta 657), reused 2381 (delta 591), pack-reused 0 (from 0)
Receiving objects: 100% (3171/3171), 32.29 MiB | 16.80 MiB/s, done.
Resolving deltas: 100% (657/657), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 70.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Step 2: Download Lily-1.5b-v0.3 from HuggingFace
from huggingface_hub import snapshot_download
import os

MODEL_DIR = '/content/Lily-1.5b-v0.3'

snapshot_download(
    repo_id = 'abhinav0231/Lily-1.5b-v0.3',
    local_dir = MODEL_DIR,
    token = os.environ.get('HF_TOKEN'),
)
print('✅ Model downloaded')

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

✅ Model downloaded


In [ ]:
import json

SYSTEM_PROMPT = (
    'You are a precise, helpful assistant. Always reason step by step '
    'inside <think> tags, then write your final answer '
    'inside <answer> tags.'
)
QWEN_DEFAULT = 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'

def patch_json_file(path):
    with open(path, 'r') as f:
        raw = f.read()

    if QWEN_DEFAULT not in raw and 'enable_thinking=True' not in raw:
        print(f'  ℹ️  Nothing to patch in {path}')
        return

    raw = raw.replace(QWEN_DEFAULT, SYSTEM_PROMPT)
    raw = raw.replace('enable_thinking=True', 'enable_thinking=False')

    with open(path, 'w') as f:
        f.write(raw)

    # Verify
    with open(path, 'r') as f:
        verified = f.read()
    assert QWEN_DEFAULT not in verified, f'❌ Old prompt still in {path}!'
    assert 'enable_thinking=True' not in verified, f'❌ enable_thinking=True still in {path}!'
    print(f'  ✅ Patched: {path}')

# Quick pre-flight check before loading tokenizer
for fname in ['tokenizer_config.json', 'tokenizer.json']:
    fpath = os.path.join(MODEL_DIR, fname)
    if os.path.exists(fpath):
        with open(fpath, 'r') as f:
            content = f.read()
        print(f'{fname}: QWEN_DEFAULT present = {QWEN_DEFAULT in content}')  # must be False
        print(f'{fname}: SYSTEM_PROMPT present = {SYSTEM_PROMPT in content}')  # must be True

import os
for fname in ['tokenizer_config.json', 'tokenizer.json']:
    fpath = os.path.join(MODEL_DIR, fname)
    if os.path.exists(fpath):
        patch_json_file(fpath)
    else:
        print(f'  ⚠️  Not found: {fpath}')

print('\n✅ All tokenizer files patched')

tokenizer_config.json: QWEN_DEFAULT present = True
tokenizer_config.json: SYSTEM_PROMPT present = False
tokenizer.json: QWEN_DEFAULT present = False
tokenizer.json: SYSTEM_PROMPT present = False
  ✅ Patched: /content/Lily-1.5b-v0.3/tokenizer_config.json
  ℹ️  Nothing to patch in /content/Lily-1.5b-v0.3/tokenizer.json

✅ All tokenizer files patched


In [ ]:
# Step 5: Convert patched HF checkpoint to F16 GGUF
!python llama.cpp/convert_hf_to_gguf.py \
    {MODEL_DIR} \
    --outtype f16 \
    --outfile /content/Lily-1.5b-v0.3-F16.gguf
print('✅ F16 GGUF created')

INFO:hf-to-gguf:Loading model: Lily-1.5b-v0.3
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:

In [ ]:
# CPU-only build — same as v0.1, fast (~3-5 min)
# !rm -rf llama.cpp/build   # clear the slow partial build
!cmake llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --config Release -j$(nproc)
print('✅ Done')

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [ ]:
import os
QUANTIZE = 'llama.cpp/build/bin/llama-quantize'
print(f'Binary exists: {os.path.exists(QUANTIZE)}')

Binary exists: True


In [ ]:
import subprocess, os

# Use the CPU-built binary from your completed cmake build
QUANTIZE = 'llama.cpp/build/bin/llama-quantize'
SRC = '/content/Lily-1.5b-v0.3-F16.gguf'

print(f'Binary exists: {os.path.exists(QUANTIZE)}')

quants = [
    ('Q4_K_M', '/content/Lily-1.5b-v0.3-Q4_K_M.gguf'),
    ('Q5_K_M', '/content/Lily-1.5b-v0.3-Q5_K_M.gguf'),
    ('Q8_0',   '/content/Lily-1.5b-v0.3-Q8_0.gguf'),
]

for qtype, out in quants:
    print(f'Quantizing → {qtype}...')
    r = subprocess.run([QUANTIZE, SRC, out, qtype], capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  ✅ {os.path.basename(out)} ({os.path.getsize(out)/1e6:.0f} MB)')
    else:
        print(f'  ❌ FAILED:\n{r.stderr[-300:]}')

Binary exists: True
Quantizing → Q4_K_M...
  ✅ Lily-1.5b-v0.3-Q4_K_M.gguf (986 MB)
Quantizing → Q5_K_M...
  ✅ Lily-1.5b-v0.3-Q5_K_M.gguf (1125 MB)
Quantizing → Q8_0...
  ✅ Lily-1.5b-v0.3-Q8_0.gguf (1647 MB)


In [ ]:
# Step 7: fast smoke test with llama-cli on CPU
import subprocess, os, textwrap

LLAMA_CLI = 'llama.cpp/build/bin/llama-cli'
GGUF_PATH = '/content/Lily-1.5b-v0.3-Q4_K_M.gguf'

assert os.path.exists(LLAMA_CLI), f"Missing binary: {LLAMA_CLI}"
assert os.path.exists(GGUF_PATH), f"Missing model: {GGUF_PATH}"

SYSTEM = (
    'You are a precise, helpful assistant. Always reason step by step '
    'inside <think> tags, then write your final answer inside <answer> tags.'
)

USER = "What is 7 plus 5? Follow the required output format exactly."

prompt = (
    f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
    f'<|im_start|>user\n{USER}<|im_end|>\n'
    f'<|im_start|>assistant\n'
)

cmd = [
    LLAMA_CLI,
    '-m', GGUF_PATH,
    '-p', prompt,
    '-n', '96',
    '-c', '512',
    '-t', '2',
    '--temp', '0.2',
    '--top-k', '20',
    '--top-p', '0.9',
    '--no-display-prompt'
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)

output = result.stdout.strip()
print("=== MODEL OUTPUT ===")
print(output[:2000])

think_count = output.count('<think>')
thinking_count = output.count('<thinking>')
answer_present = '<answer>' in output

print('\n=== FORMAT VALIDATION ===')
print(f'<think> tags:     {think_count}')
print(f'<thinking> tags:  {thinking_count}')
print(f'<answer> present: {answer_present}')

if result.stderr:
    print('\n=== STDERR TAIL ===')
    print(result.stderr[-1000:])

if think_count >= 1 and thinking_count == 0:
    print('\n✅ No stray <thinking> tag leakage — core GGUF/chat-template issue looks fixed')
else:
    print('\n⚠️ Format not ideal yet, but this may still be model behavior rather than conversion failure')

Running: llama.cpp/build/bin/llama-cli -m /content/Lily-1.5b-v0.3-Q4_K_M.gguf -p <|im_start|>system
You are a precise, helpful assistant. Always reason step by step inside <think> tags, then write your final answer inside <answer> tags.<|im_end|>
<|im_start|>user
What is 7 plus 5? Follow the required output format exactly.<|im_end|>
<|im_start|>assistant
 -n 96 -c 512 -t 2 --temp 0.2 --top-k 20 --top-p 0.9 --no-display-prompt


TimeoutExpired: Command '['llama.cpp/build/bin/llama-cli', '-m', '/content/Lily-1.5b-v0.3-Q4_K_M.gguf', '-p', '<|im_start|>system\nYou are a precise, helpful assistant. Always reason step by step inside <think> tags, then write your final answer inside <answer> tags.<|im_end|>\n<|im_start|>user\nWhat is 7 plus 5? Follow the required output format exactly.<|im_end|>\n<|im_start|>assistant\n', '-n', '96', '-c', '512', '-t', '2', '--temp', '0.2', '--top-k', '20', '--top-p', '0.9', '--no-display-prompt']' timed out after 180 seconds

In [ ]:
# Step 8: Upload all GGUFs to abhinav0231/Lily-1.5b-v0.3-GGUF
from huggingface_hub import HfApi
import os

HF_TOKEN = os.environ.get('HF_TOKEN')
REPO_ID = 'abhinav0231/Lily-1.5b-v0.3-GGUF'
api = HfApi()

api.create_repo(
    REPO_ID,
    token = HF_TOKEN,
    exist_ok = True,
    private = False,
    repo_type = 'model',
)

files = [
    ('Lily-1.5b-v0.3-F16.gguf',    '/content/Lily-1.5b-v0.3-F16.gguf'),
    ('Lily-1.5b-v0.3-Q4_K_M.gguf', '/content/Lily-1.5b-v0.3-Q4_K_M.gguf'),
    ('Lily-1.5b-v0.3-Q5_K_M.gguf', '/content/Lily-1.5b-v0.3-Q5_K_M.gguf'),
    ('Lily-1.5b-v0.3-Q8_0.gguf',   '/content/Lily-1.5b-v0.3-Q8_0.gguf'),
]

for repo_filename, local_path in files:
    if not os.path.exists(local_path):
        print(f'  ⚠️  Skipping {repo_filename} — file not found')
        continue
    size_mb = os.path.getsize(local_path) / 1e6
    print(f'Uploading {repo_filename} ({size_mb:.0f} MB)...')
    api.upload_file(
        path_or_fileobj = local_path,
        path_in_repo = repo_filename,
        repo_id = REPO_ID,
        token = HF_TOKEN,
        commit_message = 'Fix: patch chat template — disable enable_thinking + training system prompt',
    )
    print(f'  ✅ {repo_filename} uploaded')

print(f'\n🎉 Done! https://huggingface.co/{REPO_ID}')

Uploading Lily-1.5b-v0.3-F16.gguf (3094 MB)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t/Lily-1.5b-v0.3-F16.gguf:   1%|1         | 32.1MB / 3.09GB            

  ✅ Lily-1.5b-v0.3-F16.gguf uploaded
Uploading Lily-1.5b-v0.3-Q4_K_M.gguf (986 MB)...


...

  ✅ Lily-1.5b-v0.3-Q5_K_M.gguf uploaded
Uploading Lily-1.5b-v0.3-Q8_0.gguf (1647 MB)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../Lily-1.5b-v0.3-Q8_0.gguf:   0%|          | 5.91MB / 1.65GB            

  ✅ Lily-1.5b-v0.3-Q8_0.gguf uploaded

🎉 Done! https://huggingface.co/abhinav0231/Lily-1.5b-v0.3-GGUF


In [ ]:
# Step 9 (optional): Push this notebook to the base model repo for reference
# Uncomment and run after saving the notebook (File → Save)

# api.upload_file(
#     path_or_fileobj = '/content/lily_1_5B_v0_3_distill_gguf.ipynb',
#     path_in_repo = 'lily_1_5B_v0_3_distill_gguf.ipynb',
#     repo_id = 'abhinav0231/Lily-1.5b-v0.3',
#     token = os.environ.get('HF_TOKEN'),
#     commit_message = 'Fix: add GGUF conversion notebook with chat template patch',
# )
# print('✅ Notebook pushed to base model repo')